In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')

np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
model_name = 'artificial'

n_processes = 32

log_name = 'test'

with open('../transformed_event_logs/artificial_start_end_2_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['1', 'Clark', 'Jane', 'Joe', 'Karsten']
known_activities = ['DIAGNOSIS', 'QUALITY_CONTROL', 'REPAIR']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : '',
                                                        'resources' : False,
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:10<00:00, 88.75it/s] 


In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.3569348192691168035028244239')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(10152.662625216866)

In [6]:
drbart_model_path = '../../../models/advanced/'+model_name+'/resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:10<00:00, 86.65it/s] 


In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.05849629031185230448067766358')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(10151.524708464181)

In [9]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:09<00:00, 94.33it/s]


In [10]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.1719469380383942828438500385')

In [11]:
np.mean(get_pscores(likelihoods_A))

np.float64(9829.99238904516)

In [12]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:09<00:00, 95.39it/s]


In [13]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.4522402067615396120869326196')

In [14]:
np.mean(get_pscores(likelihoods_A))

np.float64(9734.259870598064)

In [15]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:09<00:00, 92.65it/s]


In [16]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.4085035028733214158647488210')

In [17]:
np.mean(get_pscores(likelihoods_A))

np.float64(9681.931454176723)

In [18]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,

                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:09<00:00, 93.98it/s] 


In [19]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.1342492128883522580504660352')

In [20]:
np.mean(get_pscores(likelihoods_A))

np.float64(8783.30160797011)

In [21]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:09<00:00, 94.40it/s] 


In [22]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.3998042006223580874673170696')

In [23]:
np.mean(get_pscores(likelihoods_A))

np.float64(8825.369303105226)

In [24]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week'
                                                                              #'(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              #'(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:08<00:00, 101.68it/s]


In [25]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-1.127785264558810297146766071')

In [26]:
np.mean(get_pscores(likelihoods_A))

np.float64(9790.815111106433)

In [29]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week_activity-count_resoure-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:09<00:00, 91.20it/s] 


In [30]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.1324688733571911012696756044')

In [31]:
np.mean(get_pscores(likelihoods_A))

np.float64(8925.03097049168)